data source : https://github.com/keeyong/beginner-spark-programming-with-pyspark/tree/main/chapter4/join_broadcast_join

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'Spark Job Visualization')
conf.set('spark.master', 'local[*]')
conf.set('spark.sql.adaptive.enabled', False)
conf.set('spark.sql.shuffle.partitions', 3)

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [ ]:
# Spark Web UI url

spark.sparkContext.uiWebUrl
# Or localhost:4040

In [2]:
df = spark.read.text('shakespeare.txt')
df.printSchema()

root
 |-- value: string (nullable = true)



In [4]:
df.show(5, truncate = False)

+------------------------------------------+
|value                                     |
+------------------------------------------+
|THE SONNETS                               |
|                                          |
|by William Shakespeare                    |
|                                          |
|From fairest creatures we desire increase,|
+------------------------------------------+
only showing top 5 rows


In [ ]:
# Job 1

import pyspark.sql.functions as f

df_count = df.select(f.explode(f.split(f.col('value'), ' ')).alias('word'))\
    .groupBy('word')\
    .count()

df_count.show()

+---------+-----+
|     word|count|
+---------+-----+
|         |  559|
|       by|   77|
|  fairest|    5|
|       we|   14|
|     rose|    3|
|    might|   18|
|    never|   13|
|     die,|    5|
|      the|  354|
|    riper|    2|
|   should|   44|
| decease,|    2|
|   tender|    7|
|     heir|    1|
|     bear|    7|
|     thou|  205|
|    eyes,|   11|
|      thy|  258|
|abundance|    4|
|      too|   14|
+---------+-----+
only showing top 20 rows


In [ ]:
# Job 2, 3

df_large = spark.read.json('large_data/')
df_small = spark.read.json('small_data/')

# Job 4

join_expr = df_large.id == df_small.id
df_join = df_large.join(df_small, join_expr, 'inner') # Shuffle Join, 하지만 용량이 작으므로 자동 Broadcast Join 적용
# df_join = df_large.join(f.broadcast(df_small), join_expr, 'inner') -> Broadcast Join

df_join.collect()